# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaryumAkram16/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** In the warehouse's `fact_content_daily_performance` table, the raw grain is one row per `report_date` × `client_hash_id` × `content_hash_id` (one day, one client's one page). For my Lane 2 analysis, I aggregate this up to **one row = one page (`content_hash_id`) for one client (`client_hash_id`), summarized over a fixed 90-day feature window**, matching the starter dataset's grain from ML-01–03.

**Time window:** I'm using **`month=2026-03`** as my mid-panel development month (per the assignment's warning — the `_sample` table is June 2026, the sealed final month, never for building label logic). My feature window is the 90 days ending at the close of March 2026; the sealed test month (June 2026) is reserved for later, honest evaluation only.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature fields** (known before any decision point):
- `gsc_impressions`, `gsc_clicks` — daily search visibility/traffic (aggregated to 90-day sums)
- `gsc_avg_position` — daily average search position (aggregated to a 90-day mean)
- `ga4_sessions` — daily analytics sessions (aggregated to a 90-day sum)

**Label / proxy field:**
- My proxy target is a **decline flag**: whether a page's clicks in the second half of the window fell versus the first half. This mirrors the starter dataset's `trend_direction == "down"`, rebuilt here from raw daily facts instead of a pre-computed bucket.

**Context fields** (used for grouping/joins, not modeling):
- `client_hash_id`, `content_hash_id`, `report_date`

**Excluded fields:**
- `ga4_data_available` when `FALSE` — rows before a client's GA4 tracking started look like "no traffic," but they're actually "not tracked yet." Excluding these prevents mistaking missing instrumentation for a real decline.
- Any FlyRank product decision flags (`health_score`, `priority_score`, `action_type`) — per the lane guide, these are never shipped in this data anyway, but I'm naming the exclusion explicitly since they'd be circular if I ever rebuilt them.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}')")

# See the real column names before writing queries against them
schema = con.sql("""
    DESCRIBE SELECT * FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    ) LIMIT 0
""")
schema.show(max_rows=50)


┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date              │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gs

In [14]:
grain_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_keys
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()
print(grain_check)
print("\nGrain confirmed if total_rows == distinct_grain_keys (no duplicate rows per key).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  distinct_grain_keys
0     9841378              9841378

Grain confirmed if total_rows == distinct_grain_keys (no duplicate rows per key).


In [15]:
slice_stats = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()
print(slice_stats)

   row_count   min_date   max_date  n_clients  n_content_items
0    9841378 2026-03-01 2026-03-31         55           331437


In [16]:
availability = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()
print(availability)
print(f"\n{availability['ga4_available_rows'][0]} of {availability['total_rows'][0]} rows "
      f"have GA4 tracking available this month — the rest are search-only.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378            413966.0

413966.0 of 9841378 rows have GA4 tracking available this month — the rest are search-only.


In [17]:
features = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) AS avg_position_90d,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_90d,
        COUNT(*) * 1.0 / 90 AS active_day_ratio
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Feature frame shape: {features.shape}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (90489, 7)


,client_hash_id,content_hash_id,impressions_90d,clicks_90d,avg_position_90d,ctr_90d,active_day_ratio
0,client_9958f0a7ae1df715,content_810cf06597918291,257.0,1.0,11.186123,0.003891,0.211111
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,180.0,1.0,8.674734,0.005556,0.077778
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,19657.0,199.0,4.532382,0.010124,0.344444
3,client_9958f0a7ae1df715,content_f5e11209b398d173,47.0,0.0,10.045000,0.000000,0.111111
4,client_9958f0a7ae1df715,content_8f3fa2db89105948,1.0,0.0,7.000000,0.000000,0.022222


1. **`impressions_90d`** — knowable at the decision moment because it's a sum of *past* daily search impressions, all dated before the review happens.
2. **`clicks_90d`** — same reasoning: purely historical daily click counts.
3. **`avg_position_90d`** — the average of daily search positions already logged; no future data involved.
4. **`ctr_90d`** — a ratio of two already-knowable past sums (clicks/impressions), so it's derived but not leaky.
5. **`active_day_ratio`** — how many of the 90 days actually had a logged row for this page; knowable immediately, since it just counts past reporting days.

In [18]:
# Build a proxy label: did clicks fall from the first half of the month to the second half?
import pandas as pd

daily = con.sql("""
    SELECT client_hash_id, content_hash_id, report_date, gsc_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE ga4_data_available IS TRUE
""").df()

daily["report_date"] = pd.to_datetime(daily["report_date"])
mid = daily["report_date"].median()
first_half = daily[daily["report_date"] <= mid].groupby(["client_hash_id","content_hash_id"])["gsc_clicks"].sum()
second_half = daily[daily["report_date"] > mid].groupby(["client_hash_id","content_hash_id"])["gsc_clicks"].sum()

change = (second_half - first_half).rename("click_change").reset_index()
label_df = features.merge(change, on=["client_hash_id","content_hash_id"], how="inner")
label_df["is_declining"] = (label_df["click_change"] < 0).astype(int)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_90d", "clicks_90d", "avg_position_90d", "ctr_90d", "active_day_ratio"]
X_honest = label_df[honest_features].fillna(0)
y = label_df["is_declining"]

tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
auc_honest = roc_auc_score(y, tree_honest.predict_proba(X_honest)[:,1])
print(f"Honest AUC (no leak): {auc_honest:.3f}")

X_leaky = X_honest.copy()
X_leaky["click_change"] = label_df["click_change"]

tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
auc_leaky = roc_auc_score(y, tree_leaky.predict_proba(X_leaky)[:,1])
print(f"Leaky AUC (with click_change as a feature): {auc_leaky:.3f}  <- jumps toward 1.0, as expected")

del X_leaky
print(f"\nKept honest number: AUC = {auc_honest:.3f}")

Honest AUC (no leak): 0.923
Leaky AUC (with click_change as a feature): 1.000  <- jumps toward 1.0, as expected

Kept honest number: AUC = 0.923


Adding `click_change` — the exact quantity my label was computed from — pushed AUC toward a near-perfect score, because the model wasn't learning a pattern, it was just reading the label back to itself. That's the leakage trap from notebook 02, reproduced here on real warehouse data. I deleted the column and kept the honest AUC as my real result.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can never tell me whether a refresh *caused* a recovery — only an experiment could show that. It's also an **unbalanced panel**: not every client has GA4 tracking from day one, so I explicitly filtered to `ga4_data_available IS TRUE` to avoid mistaking "not tracked yet" for "no traffic." Because I'm using a single mid-panel month (March 2026) rather than the full history, my feature window is short relative to the full 90-day standard used elsewhere in this track — a fuller build would pull three consecutive months to get a true 90-day window instead of one calendar month.

Concretely this month: only 413,966 of 9,841,378 daily rows (4.2%) have GA4 tracking available — meaning any feature that depends on GA4 data is only usable for a small, non-random slice of the inventory. This is why my feature frame ends up with 90,489 rows instead of the full 331,437 content items — filtering to `ga4_data_available IS TRUE` cuts out most of the panel by design, not by accident.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.